# Exercise 5: Music Classification and Statistical Testing

## Imports

In [71]:
import pathlib
from typing import Dict

import arff
import numpy as np
import pandas as pd
from sklearn import tree
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

## Constants

In [62]:
current_dir = pathlib.Path.cwd()

# Data path
data_dir_path = current_dir.joinpath("data")
train_arff_file = data_dir_path.joinpath("PopRock-TS120-FP1.arff")
test_arff_file = data_dir_path.joinpath("PopRock-TAS120-FP1.arff")

N_DECISION_TREES = 10

SEED = 0

## E1: Genre Classification with Decision Trees

In this exercise, you will train decision trees for binary genre classification between pop and rock mu-
sic. For this, you can use the sklearn implementation of sklearn.DecisionTreeClassifier.
Use the features and labels contained in the files PopRock-TS120-FP1.arff as training set, and
PopRock-TAS120-FP1.arff as test set. Note that the final category in the arff files is the label
indicator, where AG Pop corresponds to pop music, and NOT AG Pop corresponds to rock music.
Conduct the following experiment: Create 10 decision trees with 10% of the training set, randomly
chosen each time. Then, create 10 more decision trees using 50% of the training data.
Use the test set to measure the difference between the mean precision, recall and F1-scores of trees
created with more and less data.

In [63]:
def load_df(arff_file_path: pathlib.Path) -> pd.DataFrame:
    with open(arff_file_path, 'r') as f:
        print(f"Loading data from {arff_file_path}")
        dataset = arff.load(f)
        columns = [attr[0] for attr in dataset['attributes']]
        df = pd.DataFrame(dataset['data'], columns=columns)
        return df


train_df = load_df(train_arff_file)
test_df = load_df(test_arff_file)
print(train_df.head())
print(test_df.head())

target_column = train_df.columns[-1]

test_features = test_df.drop(columns=[target_column])
test_labels = test_df[target_column]

Loading data from /Users/martijuanola/git_repositories/RWTH-AIiM/exercices/e5/data/PopRock-TS120-FP1.arff
Loading data from /Users/martijuanola/git_repositories/RWTH-AIiM/exercices/e5/data/PopRock-TAS120-FP1.arff
   Mean_1(Normalized(NaN_eliminated(Zero-crossing rate)))  \
0                                           0.099468        
1                                           0.087151        
2                                           0.184871        
3                                           0.142765        
4                                           0.144051        

   Mean_1(Normalized(NaN_eliminated(Root mean square)))  \
0                                           0.363160      
1                                           0.449483      
2                                           0.416395      
3                                           0.505924      
4                                           0.264528      

   Mean_1(Normalized(NaN_eliminated(Low energy)))  \
0           

In [64]:
# 10 trees with 10% of the training set (randomly chosen)

def get_model_name(seed: int, train_size: float) -> str:
    return f"{int(train_size * 100)}_{seed}"


def get_decision_trees(train_df: pd.DataFrame, n: int = N_DECISION_TREES, seed: int = SEED, train_size: float = 0.1) -> \
        Dict[int, tree.DecisionTreeClassifier]:
    assert 0.0 <= train_size <= 1.0, f"train_size must be between 0.0 and 1.0, but is {train_size}"

    trees_dict = {}
    for seed in np.arange(seed, seed + n):
        # Split data randomly
        train_10, rest_90 = train_test_split(train_df, test_size=1 - train_size, random_state=seed)

        # Extract features and labels
        X = train_10.drop(columns=[target_column])
        Y = train_10[target_column]

        # Create a decision tree & train
        clf = tree.DecisionTreeClassifier()
        clf = clf.fit(X, Y)

        trees_dict[get_model_name(seed, train_size)] = clf

    return trees_dict

In [65]:
trees10_dict = get_decision_trees(train_df, n=N_DECISION_TREES, train_size=0.1)
trees50_dict = get_decision_trees(train_df, n=N_DECISION_TREES, train_size=0.5)
all_trees_dict = trees10_dict | trees50_dict

In [74]:
def evaluate_models(trees_dict: Dict[str, tree.DecisionTreeClassifier]) -> pd.DataFrame:
    results = []

    for model_name, clf in trees_dict.items():
        # Inference
        predicted_labels = clf.predict(test_features)

        # Calculate precision, recall and F1-score
        results.append({
            "model_name": model_name,
            "precision": (precision_score(test_labels, predicted_labels, average='weighted')),
            "recall": (recall_score(test_labels, predicted_labels, average='weighted')),
            "f1_score": (f1_score(test_labels, predicted_labels, average='weighted'))
        })

    return pd.DataFrame(results)


results_df = evaluate_models(all_trees_dict)
print(results_df)

   model_name  precision    recall  f1_score
0        10_0   0.585276  0.616667  0.579172
1        10_1   0.611607  0.625000  0.614780
2        10_2   0.539583  0.508333  0.516138
3        10_3   0.656848  0.658333  0.657545
4        10_4   0.535387  0.508333  0.516034
5        10_5   0.706094  0.700000  0.702273
6        10_6   0.606609  0.608333  0.607430
7        10_7   0.555761  0.608333  0.538978
8        10_8   0.772556  0.750000  0.753836
9        10_9   0.588443  0.625000  0.558596
10       50_0   0.617500  0.608333  0.611887
11       50_1   0.737090  0.741667  0.732506
12       50_2   0.754252  0.758333  0.753530
13       50_3   0.693750  0.691667  0.654396
14       50_4   0.745726  0.750000  0.745879
15       50_5   0.644075  0.633333  0.637153
16       50_6   0.726130  0.716667  0.719618
17       50_7   0.676484  0.675000  0.675695
18       50_8   0.714361  0.716667  0.715311
19       50_9   0.737500  0.741667  0.738235


## E2: Statistical Testing

Test whether the precision, recall and F1-scores of your decision trees are normally distributed. Then,
use non-parametric statistical testing to see if the differences in performance are statistically signifi-
cant (α= 0.05).

## E3: Feature Evaluation

Visualise one of your trees for the low-data case and one for the large-data case, using the tutorial
given by sklearn.
Finally, investigate the features your decision trees have chosen to classify the data. Which features
are chosen most commonly?